### Imports

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm

### Parameters

In [2]:
alpha = 0.95
k = norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(k) 

2.0627128075074257


In [3]:
df = pd.read_csv("selected_monthly_return.csv", encoding="utf-8-sig")
tickers = df["Ticker"].tolist()
returns = df.drop(columns=["PERMNO", "Ticker", "average_monthly_return"])
mu = df["average_monthly_return"].to_numpy(dtype=float)

### Covariance Matrix

In [4]:
cov_df = pd.read_csv("covariance_matrix.csv", skiprows=1, encoding="utf-8-sig")

Sigma = cov_df.drop(columns=["PERMNO", "Ticker"]).to_numpy(dtype=float)

eigvals, eigvecs = np.linalg.eigh(Sigma)
A = np.sqrt(np.clip(eigvals, 0, None))[:, None] * eigvecs.T

### Model

In [5]:
# model
m = gp.Model("portfolio")

Set parameter Username
Set parameter LicenseID to value 2866621
Academic license - for non-commercial use only - expires 2027-09-22


### Decision Variables

In [6]:
# weights
w = m.addMVar(len(tickers), vtype=GRB.CONTINUOUS, name="w")
# standard deviation of portfolio
s = m.addVar(vtype=GRB.CONTINUOUS, name="s")

### Constraints

In [7]:
# constraint 1
m.addConstr(w.sum() == 1, name="full_investment")
# constraint 2
m.addConstr(w <= 0.2, name="max_weight")

<MConstr (37,) *awaiting model update*>

In [8]:
# constraint 3
benchmark = pd.read_csv("SP500_monthly_202001_202603.csv", encoding="utf-8-sig")
benchmark["Month"] = pd.to_datetime(benchmark["Month"])
b = benchmark.loc[benchmark["Month"].between("2021-01-01", "2025-12-31"), "MonthlyReturn"].mean()
m.addConstr(w @ mu >= b, name="minimum_return")

<MConstr () *awaiting model update*>

In [9]:
# constraint 4
y = m.addMVar(37, lb=-GRB.INFINITY, name="y") # helper variable for finding s 
m.addConstr(y == A @ w)
m.addGenConstrNorm(s, y, 2.0) # constraint to find s

<gurobi.GenConstr *Awaiting Model Update*>

### Objective Function

In [10]:
m.setObjective(-1 * w @ mu + k * s, gp.GRB.MINIMIZE)

### Results

In [11]:
m.optimize()

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 PRO 5850U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 76 rows, 75 columns and 1517 nonzeros
Model fingerprint: 0x3349a64e
Model has 1 general constraint
Variable types: 75 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [2e-03, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-02, 1e+00]
Presolve removed 37 rows and 0 columns
Presolve time: 0.03s
Presolved: 40 rows, 112 columns, 1512 nonzeros
Presolved model has 37 quadratic constraint(s)
Variable types: 112 continuous, 0 integer (0 binary)
Root relaxation presolve time: 0.06s
Root relaxation presolved: 118 rows, 186 columns, 1738 nonzeros
Root relaxation presolved model has 37 second-order cone constraints
Root barrier log...

Ordering time: 0.00

In [12]:
portfolio = pd.DataFrame({"Ticker": tickers, "Optimized Weight": w.X})
print(portfolio.to_string(index=False, formatters={"Optimized Weight": "{:.6%}".format}))

Ticker Optimized Weight
  COHR        0.000000%
    GE        0.000001%
   FNV        0.000004%
    UI        0.000001%
  BABA        3.758587%
  LITE        0.000000%
  CVNA        0.000000%
   CVS        6.786247%
   NET        0.000000%
  PLTR        0.000000%
  SOFI        0.000000%
   NEM        0.000000%
   CAH       19.995638%
   GLW        0.434647%
   HWM        0.000000%
  KLAC        0.000000%
  LRCX        0.000000%
    MU        0.000000%
  INTC        0.000000%
   AMD        0.000000%
   WDC        0.000000%
     B        0.000001%
  IDXX        0.000000%
   AEM        0.000000%
  ESLT       19.999989%
  CIEN        0.000000%
   APH        0.000002%
   FIX        7.687506%
    MT        0.000000%
   CLS        0.000001%
  INSM        4.128126%
    DB        8.014004%
   STX        0.000000%
   KGC        0.000000%
   NRG        0.000001%
   WPM       13.996938%
  AVGO       15.198303%


In [13]:
expected_return = mu @ w.X

print(f"Expected portfolio return for January 2026: {expected_return:.6%}")
print(f"Portfolio monthly standard deviation: {s.X:.6%}")
print(f"Portfolio monthly 95% CVaR: {m.ObjVal:.6%}")

Expected portfolio return for January 2026: 2.997920%
Portfolio monthly standard deviation: 4.328212%
Portfolio monthly 95% CVaR: 5.929938%


In [14]:
print(f"Historical benchmark mean monthly return b: {b:.6%}") # 2021-2025
print(f"Expected portfolio monthly return: {expected_return:.6%}")
print(f"Excess return: {(expected_return - b):.6f}")
print(f"Minimum-return constraint satisfied: {'Yes' if expected_return >= b - 1e-6 else 'No'}")

Historical benchmark mean monthly return b: 1.099027%
Expected portfolio monthly return: 2.997920%
Excess return: 0.018989
Minimum-return constraint satisfied: Yes
